# 04 · Retrieve — 03 Dedupe and Merge

**Everything here is deterministic, offline, and uses synthetic duplicate records constructed inline -- no network call and no API key needed.**

Implements `merge_and_deduplicate_sources` and its three normalizers
(`_norm_doi`, `_norm_pmcid`, `_norm_title_key`) plus the fuzzy-title matcher
(`_title_fuzzy_dup`).

**In → out:** the per-source lists notebook `02` printed → one merged list
with duplicates collapsed, in first-seen order, via a three-rung ladder:
**DOI → PMCID → fuzzy title at a 0.92 similarity threshold.**

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `_norm_doi` | Lowercases a DOI and strips a `https://doi.org/` prefix. | `_norm_doi("https://doi.org/10.1234/BURN.2020.0099")` → `"10.1234/burn.2020.0099"` |
| `_norm_pmcid` | Canonicalizes a PMC id; rejects anything that still doesn't match `PMC\d+`. | `_norm_pmcid("7654321")` → `"PMC7654321"` |
| `_norm_title_key` | Collapses whitespace/case for exact-match comparison. | `_norm_title_key("  A   Title ")` → `"a title"` |
| `_title_fuzzy_dup` | Fuzzy title match against a list of already-seen titles, at a 0.92 threshold. | `_title_fuzzy_dup(title, seen_norm)` → `True`/`False` |
| `merge_and_deduplicate_sources` | Runs the full DOI → PMCID → fuzzy-title ladder across ordered source lists. | `merge_and_deduplicate_sources([source_a, source_b])` |


In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()

## Step 1 — the three normalizers

- `_norm_doi` -- lowercases and strips a `https://doi.org/` prefix, so the
  same DOI written two different ways still compares equal.
- `_norm_pmcid` -- validates and canonicalizes a PMC id: a bare numeric string
  becomes `PMC<digits>`, and anything that still doesn't match `PMC\d+` is
  rejected back to `""` (a malformed id must never falsely collide with a
  real one).
- `_norm_title_key` -- collapses whitespace and case for an exact-match
  comparison, used both directly and as the input to the fuzzy matcher below.

In [ ]:
import re
from difflib import SequenceMatcher

_PMC_VALID = re.compile(r"^PMC\d+$", re.IGNORECASE)


def _norm_doi(d: str) -> str:
    x = (d or "").strip().lower()
    if x.startswith("https://doi.org/"):
        x = x.replace("https://doi.org/", "").strip()
    return x


def _norm_pmcid(p: str) -> str:
    raw = (p or "").strip().upper()
    if not raw:
        return ""
    if not raw.startswith("PMC") and raw.isdigit():
        raw = f"PMC{raw}"
    return raw if _PMC_VALID.match(raw) else ""


def _norm_title_key(t: str) -> str:
    return " ".join((t or "").lower().split())

## Step 2 — synthetic duplicate records, built to exercise every rung

Nine made-up "papers" (no real bibliographic data) split across two synthetic
sources, built to exercise every rung of the ladder plus the boundary the
0.92 threshold sits on:

| Pair | Rung that should catch it | Shared DOI? | Shared PMCID? | Title similarity |
|---|---|---|---|---|
| Burn-excision | **DOI** | yes (different case + `doi.org/` prefix) | no | n/a |
| NPWT trial | **PMCID** | no | yes (bare digits vs `PMC`-prefixed) | 0.48 -- title alone would *not* have caught this |
| NPUAP staging | **fuzzy title** | no | no | 0.98 -- colon vs dash punctuation only |
| DIEP flap | *(none -- should survive as two records)* | no | no | 0.86 -- close, but under 0.92 |

A fifth, fully unique record (with an `abstract` but no `text`, to exercise
the abstract→text fallback) rounds out the set. Defined here, before any
rung of the ladder is tested, so each rung below is tested against real
records rather than invented on the spot.

In [ ]:
source_a = [
    {
        "title": "Early tangential excision timing in deep partial thickness burns",
        "text": "Cohort study of excision timing and blood loss.",
        "doi": "https://doi.org/10.1234/BURN.2020.0099",
        "pmcid": "",
        "source": "pubmed",
    },
    {
        "title": "Negative pressure wound therapy in diabetic foot ulcers: a randomized trial",
        "text": "RCT of NPWT versus standard dressings in diabetic foot ulcers.",
        "doi": "",
        "pmcid": "7654321",
        "source": "pubmed",
    },
    {
        "title": "NPUAP pressure injury staging system: classification and clinical application",
        "text": "Reference summary of the staging system.",
        "doi": "",
        "pmcid": "",
        "source": "pubmed",
    },
    {
        "title": "DIEP flap versus TRAM flap for breast reconstruction: complication rates",
        "text": "Comparative outcomes review of abdominal-based free flaps.",
        "doi": "",
        "pmcid": "",
        "source": "pubmed",
    },
]

source_b = [
    {
        "title": "Early Tangential Excision Timing In Deep Partial Thickness Burns",
        "text": "Duplicate listing from a second source, same paper, different case.",
        "doi": "10.1234/burn.2020.0099",
        "pmcid": "",
        "source": "openalex",
    },
    {
        "title": "NPWT for diabetic foot ulceration: 12-week outcomes",  # same trial, PMCID matches, title doesn't
        "text": "Same trial, indexed independently under a shorter title.",
        "doi": "",
        "pmcid": "PMC7654321",
        "source": "s2",
    },
    {
        "title": "NPUAP pressure injury staging system - classification and clinical application",  # dash, not colon
        "text": "Same reference summary, re-punctuated by a different indexer.",
        "doi": "",
        "pmcid": "",
        "source": "s2",
    },
    {
        "title": "DIEP flap versus TRAM flap for breast reconstruction: complication rates "
        "at five-year follow-up",  # related, NOT the same record -- should survive
        "text": "A longer-follow-up companion study -- a different paper, correctly kept separate.",
        "doi": "",
        "pmcid": "",
        "source": "vectors",
    },
    {
        "title": "Parkland formula for fluid resuscitation in burns: evidence and alternatives",
        "abstract": "Review of the Parkland formula and documented over-resuscitation risk.",
        "doi": "",
        "pmcid": "",
        "source": "vectors",
    },
]


## Step 3 — count the raw input before any merge runs

In [ ]:
doc_lists = [source_a, source_b]
before_count = sum(len(d) for d in doc_lists)
print(f"before: {before_count} raw records across {len(doc_lists)} synthetic sources")

## Step 4 — the DOI rung, tested alone

Before assembling the full ladder, check that DOI matching alone catches the
burn-excision pair: same DOI, written two different ways (case, and a
`https://doi.org/` prefix on one side).

In [ ]:
doi_a = _norm_doi(source_a[0]["doi"])
doi_b = _norm_doi(source_b[0]["doi"])
print(f"source_a DOI normalized: {doi_a!r}")
print(f"source_b DOI normalized: {doi_b!r}")
print(f"DOI rung alone catches the burn-excision pair: {doi_a == doi_b}")

## Step 5 — the PMCID rung, tested alone

The NPWT trial pair shares no DOI, and its titles alone sit at only 0.48
similarity -- title matching would miss it entirely. Check PMCID matching
catches it on its own, independent of the other two rungs.

In [ ]:
pmc_a = _norm_pmcid(source_a[1]["pmcid"])
pmc_b = _norm_pmcid(source_b[1]["pmcid"])
title_ratio = SequenceMatcher(None, _norm_title_key(source_a[1]["title"]), _norm_title_key(source_b[1]["title"])).ratio()
print(f"source_a PMCID normalized: {pmc_a!r}")
print(f"source_b PMCID normalized: {pmc_b!r}")
print(f"title similarity alone: {title_ratio:.3f} (well under the 0.92 threshold)")
print(f"PMCID rung alone catches the NPWT pair: {pmc_a == pmc_b}")

## Step 6 — the fuzzy-title rung, tested alone, at the 0.92 boundary

"0.92" is just a number in a docstring until you see what actually sits on
either side of it. The three pairs below are exactly the ones the merge in
Step 8 has to resolve: the NPUAP pair (should dedup), the PMCID pair's
titles (shown again here to confirm title-only similarity would NOT have
caught it), and the DIEP flap pair (a close call that should be kept
separate).

In [ ]:
pairs = [
    (
        "PMCID pair's titles (title alone would NOT catch this)",
        "Negative pressure wound therapy in diabetic foot ulcers: a randomized trial",
        "NPWT for diabetic foot ulceration: 12-week outcomes",
    ),
    (
        "NPUAP pair (fuzzy title IS what catches this)",
        "NPUAP pressure injury staging system: classification and clinical application",
        "NPUAP pressure injury staging system - classification and clinical application",
    ),
    (
        "DIEP flap pair (boundary -- correctly kept separate)",
        "DIEP flap versus TRAM flap for breast reconstruction: complication rates",
        "DIEP flap versus TRAM flap for breast reconstruction: complication rates at five-year follow-up",
    ),
]

rows = []
for label, t1, t2 in pairs:
    ratio = SequenceMatcher(None, _norm_title_key(t1), _norm_title_key(t2)).ratio()
    rows.append((label, f"{ratio:.3f}", "would DEDUP on title alone" if ratio >= 0.92 else "kept separate"))

nbio.table(rows, headers=("pair", "ratio", "outcome at threshold 0.92"))

## Step 7 — `merge_and_deduplicate_sources` — all three rungs together

Ordered source lists go in; one deduplicated list comes out, **in first-seen
order** -- a document's position is decided by which source found it first,
not by any score. Note the abstract-as-text fallback at the end: a document
that has an `abstract` field but no `text` gets one, since RCS (notebook `04`)
only ever reads `text`. Each rung above was just shown catching its pair
alone; this is the same three checks, run together, in the order
DOI → PMCID → fuzzy title.

In [ ]:
def _title_fuzzy_dup(title: str, seen_norm: list[str], threshold: float = 0.92) -> bool:
    nt = _norm_title_key(title)
    if len(nt) < 12:
        return False
    for existing in seen_norm:
        if SequenceMatcher(None, nt, existing).ratio() >= threshold:
            return True
    return False


def merge_and_deduplicate_sources(doc_lists: list[list[dict]]) -> list[dict]:
    seen_doi: set[str] = set()
    seen_pmc: set[str] = set()
    seen_titles_norm: list[str] = []
    out: list[dict] = []

    for docs in doc_lists:
        for d in docs:
            doi = _norm_doi(d.get("doi") or "")
            pmc = _norm_pmcid(d.get("pmcid") or "")
            title = (d.get("title") or "").strip()
            nt = _norm_title_key(title)

            if doi and doi in seen_doi:
                continue
            if pmc and pmc in seen_pmc:
                continue
            if nt and len(nt) >= 12:
                if nt in seen_titles_norm:
                    continue
                if _title_fuzzy_dup(title, seen_titles_norm):
                    continue

            if doi:
                seen_doi.add(doi)
            if pmc:
                seen_pmc.add(pmc)
            if nt and len(nt) >= 12:
                seen_titles_norm.append(nt)

            d2 = dict(d)
            if not d2.get("text") and d2.get("abstract"):
                d2["text"] = d2["abstract"]
            out.append(d2)
    return out

## Step 8 — run the full merge and look at real output

In [ ]:
merged = merge_and_deduplicate_sources(doc_lists)
print(f"after:  {len(merged)} records\n")
nbio.table(
    [(d["source"], d["doi"] or "-", d["pmcid"] or "-", d.get("text", "")[:35], d["title"][:55]) for d in merged],
    headers=("source", "doi", "pmcid", "text (abstract fallback applied?)", "title"),
)

## Wrap-up

Nine raw records across two synthetic sources merge down to six: the
burn-excision pair collapsed on **DOI**, the NPWT pair collapsed on
**PMCID** even though their titles alone sit at 0.48 similarity (well under
the fuzzy threshold -- the PMCID rung is doing real, independent work here),
and the NPUAP pair collapsed on **fuzzy title** at 0.98 with no shared
identifier at all. The two DIEP flap records, at 0.86 similarity, correctly
survive as two separate papers -- a close call, and a fair one: a hard
threshold has to draw the line somewhere, and a genuine five-year follow-up
study of the same cohort can look almost exactly like this from the outside.

Next: `04-llm-chunk-scoring.ipynb` -- what happens to this merged list before
it's ranked: an LLM scores every surviving candidate for relevance, and
anything below the threshold is dropped.